In [2]:
model     = HCAMCapsNet().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
criterion = CombinedLoss(aux_weight=0.4, label_smoothing=0.1)

NUM_EPOCHS  = 60
WARMUP      = 5

scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2, eta_min=1e-6
)

# ── epoch-wise history ───────────────────────────────────────────────────────
history = {
    "epoch":        [],
    "lr":           [],
    "train_loss":   [],
    "train_fine":   [],
    "val_fine":     [],
    "val_coarse":   [],
}

best_acc  = 0.0
HDR = (f"{'Ep':>3} | {'LR':>8} | {'TrLoss':>7} | "
       f"{'TrFine':>7} | {'VaFine':>7} | {'VaCoarse':>8} |")
SEP = "-" * len(HDR)
print(SEP)
print(HDR)
print(SEP)

import time

for epoch in range(NUM_EPOCHS):
    t0 = time.time()

    # ---- linear warmup --------------------------------------------------
    if epoch < WARMUP:
        for pg in optimizer.param_groups:
            pg["lr"] = 3e-4 * (epoch + 1) / WARMUP

    # ---- training -------------------------------------------------------
    model.train()
    run_loss = correct_fine = total_tr = 0

    for X, f, c in train_loader:
        X, f, c = X.to(device), f.to(device), c.to(device)

        X_m, f_a, c_a, f_b, c_b, lam = mixup_batch(X, f, c)

        optimizer.zero_grad()
        _, _, cp, fp = model(X_m)

        loss = lam * criterion(cp, fp, c_a, f_a) +                (1 - lam) * criterion(cp, fp, c_b, f_b)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        run_loss     += loss.item()
        # Use the majority mixup label for training accuracy
        correct_fine += (fp.argmax(1) == f_a).sum().item()
        total_tr     += f_a.size(0)

    if epoch >= WARMUP:
        scheduler.step()

    train_loss = run_loss / len(train_loader)
    train_fine = correct_fine / total_tr

    # ---- validation -----------------------------------------------------
    model.eval()
    correct_fine_v = correct_coarse_v = total_v = 0

    with torch.no_grad():
        for X, f, c in test_loader:
            X, f, c = X.to(device), f.to(device), c.to(device)
            _, _, cp, fp = model(X)
            correct_fine_v   += (fp.argmax(1) == f).sum().item()
            correct_coarse_v += (cp.argmax(1) == c).sum().item()
            total_v          += f.size(0)

    val_fine   = correct_fine_v   / total_v
    val_coarse = correct_coarse_v / total_v
    cur_lr     = optimizer.param_groups[0]["lr"]
    elapsed    = time.time() - t0

    # ── record history ───────────────────────────────────────────────────
    history["epoch"].append(epoch + 1)
    history["lr"].append(cur_lr)
    history["train_loss"].append(train_loss)
    history["train_fine"].append(train_fine)
    history["val_fine"].append(val_fine)
    history["val_coarse"].append(val_coarse)

    marker = " <<" if val_fine > best_acc else ""
    if val_fine > best_acc:
        best_acc = val_fine
        torch.save(model.state_dict(), "hcam_capsnet_best.pth")

    print(f"{epoch+1:>3} | {cur_lr:>8.2e} | {train_loss:>7.4f} | "
          f"{train_fine:>7.4f} | {val_fine:>7.4f} | {val_coarse:>8.4f} |"
          f"  {elapsed:5.1f}s{marker}")

print(SEP)
print(f"Best validation fine-accuracy: {best_acc:.4f}")

# ── summary table ────────────────────────────────────────────────────────────
print()
print("=== EPOCH-WISE SUMMARY ===")
print(f"{'Ep':>3}  {'LR':>9}  {'TrLoss':>8}  {'TrFineAcc':>10}  {'ValFineAcc':>11}  {'ValCoarseAcc':>13}")
for i in range(len(history["epoch"])):
    print(f"{history['epoch'][i]:>3}  "
          f"{history['lr'][i]:>9.2e}  "
          f"{history['train_loss'][i]:>8.4f}  "
          f"{history['train_fine'][i]:>10.4f}  "
          f"{history['val_fine'][i]:>11.4f}  "
          f"{history['val_coarse'][i]:>13.4f}")
